# B1 · How many components? PCA in a pipeline on the digits

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Choose the number of components of a PCA in a `StandardScaler` → `PCA` → `LogisticRegression` `Pipeline` with cross-validation, compare with the model without reduction, and find out that **standardizing the pixels changes how many components are needed**: `PCA(0.95)` chooses
a different number depending on the scaler (and from one fold to another).

## Dataset

`load_digits`: 1797 8×8 images, 64 pixels (0-16), 10 classes. Validation: `StratifiedKFold(5, shuffle=True, random_state=0)`, accuracy.

## Educational purpose

Treat `k` as a hyperparameter (like `C`), read the accuracy curve as a function of `k` with an explicit choice rule and understand that `n_components=0.95` is not a number: it is a rule, and the number it gives depends on the data and on the preprocessing.

## Task

1. Compute `acc_k` (`pd.Series` indexed by `k`) with the cross-validated accuracy of `make_pipeline(StandardScaler(), PCA(k), LogisticRegression(max_iter=3000))` for `k` in `[2, 5, 10, 15, 20, 30, 40, 64]`.
2. Save `acc_without_pca` (same pipeline without PCA) and `rule_k`: the **smallest** `k` whose accuracy is within **0.01** of the maximum of `acc_k`. Save `rule_k_loss = acc_without_pca - acc_k[rule_k]`.
3. With `cross_validate(..., return_estimator=True)` on the pipeline `StandardScaler` → `PCA(0.95)` → logistic, save `scaled_k_per_fold` (list of `n_components_` per fold). Repeat **without** `StandardScaler` (`PCA(0.95)` on the raw pixels) and save `raw_k_per_fold` and `raw_acc` (mean accuracy).
4. Save `k_difference`: mean of `scaled_k_per_fold` minus mean of `raw_k_per_fold`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
ks = [2, 5, 10, 15, 20, 30, 40, 64]

## Your work

In [ ]:
acc_k = pd.Series(dtype=float)
acc_k

In [ ]:
acc_without_pca = None
rule_k = None
rule_k_loss = None

In [ ]:
scaled_k_per_fold = None
raw_k_per_fold = None
raw_acc = None
k_difference = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='To get the components per fold: `r = cross_validate(pipe, X, y, cv=cv, return_estimator=True); [e["pca"].n_components_ for e in r["estimator"]]` (use a name for the step, for example `Pipeline([("pca", PCA(0.95)), ...])`, or `e[1].n_components_` with `make_pipeline`).'):
    check(list(acc_k.index) == ks and acc_k[2] < 0.6 and acc_k[64] > 0.96, "curve over eight values of k: with 2 components below 0.6, with 64 above 0.96", "pipeline with StandardScaler, PCA(k), logistic")
    check(acc_k.is_monotonic_increasing, "the accuracy grows with k", "more components, more information")
    check(acc_without_pca > 0.96 and abs(acc_without_pca - acc_k[64]) < 1e-9, "with 64 components (no reduction) the accuracy is the one without PCA: a rotation does not change the logistic", "PCA(64) is just a rotation of the axes")
    check(rule_k == min(k for k in ks if acc_k[k] >= acc_k.max() - 0.01) and 30 <= rule_k <= 40 and 0 < rule_k_loss < 0.012, "k of the rule: 30-40 components out of 64, with a loss below 1.2 points", "the smallest k within 0.01 of the maximum")
    check(len(scaled_k_per_fold) == 5 and all(35 <= k <= 45 for k in scaled_k_per_fold) and all(24 <= k <= 32 for k in raw_k_per_fold), "PCA(0.95): 35-45 components with the scaler, 24-32 without", "cross_validate(..., return_estimator=True) and pipeline[k].n_components_")
    check(k_difference > 8 and raw_acc > 0.94, "the scaler requires over 8 more components for the same 95% of variance; without the scaler the accuracy stays above 0.94", "means of k_per_fold")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_choosing_k_in_a_pipeline_on_digits.ipynb`. Open it after you pass the check, or after you have really tried.